In [ ]:
from BattleStateTracker import BattleStateTracker 
import json
from StateVectorization import iter_turn_examples_both_players, build_move_vocab, vectorize_dataset
from TrainingSplit import group_split_by_battle_id, ingest_battles_to_examples
from StaticDex import StaticDex

In [ ]:
# -----------------------------
# Run on your single battle file
# -----------------------------
path = "./gen9randombattle-2390494424.json"
with open(path, "r") as f:
    battle = json.load(f)

tracker = BattleStateTracker()
examples_both = list(iter_turn_examples_both_players(tracker, battle))

print("num examples (both players, move-only):", len(examples_both))
print("sample:", examples_both[0]["player"], examples_both[0]["turn_number"], examples_both[0]["action"])

move_vocab = build_move_vocab(examples_both, min_count=1)
print("num move classes (including <UNK>):", len(move_vocab))
print("top 10 vocab items:", list(move_vocab.items())[:10])

X, y = vectorize_dataset(examples_both, move_vocab)
print("X dim:", len(X), "x", len(X[0]))
print("y dim:", len(y))
print("first y:", y[0])
print(y)


In [ ]:
import numpy as np

# Convert lists -> numpy arrays (this enables fancy indexing)
X = np.asarray(X, dtype=np.float32)
y = np.asarray(y, dtype=np.int64)
train_idx, val_idx = group_split_by_battle_id(examples_both, val_ratio=0.2, seed=42)

print(examples_both[0:2])
X_train, y_train = X[train_idx], y[train_idx]
X_val, y_val     = X[val_idx], y[val_idx]

print("X_train:", X_train.shape, "X_val:", X_val.shape)
print("y_train:", y_train.shape, "y_val:", y_val.shape)

print("num classes:", len(move_vocab))


In [ ]:
import kagglehub
from pathlib import Path

path = kagglehub.dataset_download("thephilliplin/pokemon-showdown-battles-gen9-randbats")
print("Dataset path:", path)

root = Path(path)
json_paths = sorted([str(p) for p in root.rglob("*.json")])
print("Num JSON files found:", len(json_paths))
print("Example file:", json_paths[0] if json_paths else None)


In [ ]:
examples_both = ingest_battles_to_examples(tracker, json_paths, max_battles=5000, verbose_every=200)

print("Total examples:", len(examples_both))
print("Unique battles:", len(set(e["battle_id"] for e in examples_both)))

In [ ]:
move_vocab = build_move_vocab(examples_both, min_count=1)
print("num move classes (including <UNK>):", len(move_vocab))
print("top 10 vocab items:", list(move_vocab.items())[:10])

X, y = vectorize_dataset(examples_both, move_vocab)
print("X dim:", len(X), "x", len(X[0]))
print("y dim:", len(y))
print("first y:", y[0])
print(y)


In [ ]:
import numpy as np

# Convert lists -> numpy arrays (this enables fancy indexing)
X = np.asarray(X, dtype=np.float32)
y = np.asarray(y, dtype=np.int64)
train_idx, val_idx = group_split_by_battle_id(examples_both, val_ratio=0.2, seed=42)

X_train, y_train = X[train_idx], y[train_idx]
X_val, y_val     = X[val_idx], y[val_idx]

print("X_train:", X_train.shape, "X_val:", X_val.shape)
print("y_train:", y_train.shape, "y_val:", y_val.shape)

print("num classes:", len(move_vocab))

In [ ]:
# SETUP MODEL
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

num_classes = len(move_vocab)
input_dim = X_train.shape[1]

model = keras.Sequential([
    layers.Input(shape=(input_dim,)),
    layers.Dense(256, activation="relu"),
    layers.Dropout(0.1),
    layers.Dense(256, activation="relu"),
    layers.Dropout(0.1),
    layers.Dense(256, activation="relu"),
    layers.Dropout(0.1),
    layers.Dense(num_classes)  # logits
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=[
        keras.metrics.SparseCategoricalAccuracy(name="top1"),
        keras.metrics.SparseTopKCategoricalAccuracy(k=3, name="top3"),
    ],
)

model.summary()


In [ ]:
# TRAAAAAAAAAAAAAAAAAAAIN 
callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_top3", patience=3, mode="max", restore_best_weights=True)
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=5,
    batch_size=256,
    # batch_size=512,
    # callbacks=callbacks,
    verbose=1,
)


In [ ]:
import numpy as np

# Build an inverse vocab: class_index -> move_id
idx_to_move = {i: m for m, i in move_vocab.items()}

def show_predictions(model, X, y=None, n=5, k=3, seed=0):
    rng = np.random.default_rng(seed)
    n = min(n, len(X))
    sample_idxs = rng.choice(len(X), size=n, replace=False)

    probs = model.predict(X[sample_idxs], verbose=0)  # shape: (n, num_classes)

    for row_i, ex_i in enumerate(sample_idxs):
        p = probs[row_i]

        topk_idx = np.argsort(p)[-k:][::-1]  # highest first
        topk = [(idx_to_move.get(int(ci), "<UNK>"), float(p[ci])) for ci in topk_idx]

        top1_move, top1_prob = topk[0]

        print(f"\nExample idx={ex_i}")
        print(f"Top-1: {top1_move}  (p={top1_prob:.3f})")

        print(f"Top-{k}:")
        for rank, (mv, pr) in enumerate(topk, start=1):
            print(f"  {rank}. {mv:20s} p={pr:.3f}")

        if y is not None:
            true_idx = int(y[ex_i])
            true_move = idx_to_move.get(true_idx, "<UNK>")
            # rank of the true class among all classes (1 = best)
            true_rank = int(np.sum(p > p[true_idx]) + 1)
            print(f"True:  {true_move}  (rank {true_rank}, p={float(p[true_idx]):.3f})")

# Usage:
show_predictions(model, X_val, y=y_val, n=5, k=3, seed=42)


In [ ]:
import numpy as np

idx_to_move = {i: m for m, i in move_vocab.items()}

def _softmax(x: np.ndarray) -> np.ndarray:
    x = x - np.max(x)
    e = np.exp(x)
    return e / np.sum(e)

def _get_active_info(state, side_key):
    side = state.get(side_key, {}) or {}
    uid = side.get("active_uid")
    if not uid:
        return None, None, []
    mon = (state.get("mons", {}) or {}).get(uid, {}) or {}
    return uid, mon.get("species"), mon.get("observed_moves", []) or []

def show_predictions_with_context(
    model,
    X_val,
    y_val=None,
    examples_val=None,
    n=5,
    k=5,
    seed=42,
    show_softmax=True,
    show_observed_moves=True,
):
    rng = np.random.default_rng(seed)
    n = min(n, len(X_val))
    sample_idxs = rng.choice(len(X_val), size=n, replace=False)

    logits_batch = model.predict(X_val[sample_idxs], verbose=0)  # logits (given your from_logits setup)

    for row_i, local_i in enumerate(sample_idxs):
        logits = logits_batch[row_i].astype(np.float32)

        print("\n==============================")

        # ---- context ----
        if examples_val is not None:
            ex = examples_val[local_i]
            battle_id = ex.get("battle_id", "unknown_battle")
            turn_number = ex.get("turn_number", ex.get("turn_index", "?"))
            perspective = ex.get("player", "?")

            state = ex.get("state", {}) or {}
            p1_uid, p1_species, p1_obs = _get_active_info(state, "p1")
            p2_uid, p2_species, p2_obs = _get_active_info(state, "p2")

            print(f"Battle: {battle_id}")
            print(f"Replay: https://replay.pokemonshowdown.com/{battle_id}")
            print(f"Turn:   {turn_number} (perspective={perspective})")
            print(f"Facing: {p1_species or '?'} ({p1_uid or '?'})  vs  {p2_species or '?'} ({p2_uid or '?'})")

            if show_observed_moves:
                if p1_uid:
                    print(f"p1 active observed_moves: {p1_obs}")
                if p2_uid:
                    print(f"p2 active observed_moves: {p2_obs}")
        # ---------------

        # top-k by logits
        topk_idx = np.argsort(logits)[-k:][::-1]
        topk_moves = [idx_to_move.get(int(ci), "<UNK>") for ci in topk_idx]
        topk_logits = [float(logits[ci]) for ci in topk_idx]

        print("\nPredictions (logits):")
        for rank, (mv, lg) in enumerate(zip(topk_moves, topk_logits), start=1):
            print(f"  {rank}. {mv:20s} logit={lg:.3f}")

        if show_softmax:
            probs = _softmax(logits)
            print("\nPredictions (softmax over all moves):")
            for rank, ci in enumerate(topk_idx, start=1):
                mv = idx_to_move.get(int(ci), "<UNK>")
                print(f"  {rank}. {mv:20s} p={float(probs[ci]):.3f}")

        if y_val is not None:
            true_idx = int(y_val[local_i])
            true_move = idx_to_move.get(true_idx, "<UNK>")
            true_rank = int(np.sum(logits > logits[true_idx]) + 1)
            print(f"\nTrue move: {true_move} (rank {true_rank}, logit={float(logits[true_idx]):.3f})")
examples_val = [examples_both[i] for i in val_idx]
show_predictions_with_context(model, X_val, y_val=y_val, examples_val=examples_val, n=5, k=5, seed=45)


In [ ]:
import numpy as np
from collections import defaultdict

idx_to_move = {i: m for m, i in move_vocab.items()}

def _softmax(x: np.ndarray) -> np.ndarray:
    x = x - np.max(x)
    e = np.exp(x)
    return e / np.sum(e)

def _get_active_info(state, side_key):
    side = state.get(side_key, {}) or {}
    uid = side.get("active_uid")
    if not uid:
        return None, None, []
    mon = (state.get("mons", {}) or {}).get(uid, {}) or {}
    return uid, mon.get("species"), mon.get("observed_moves", []) or []

def show_battle_window_alternating(
    model,
    X_val,
    y_val,
    examples_val,
    battle_id=None,
    max_turns=12,
    k=5,
    seed=42,
    show_softmax=True,
    show_observed_moves=True,
):
    rng = np.random.default_rng(seed)

    by_battle = defaultdict(list)
    for i, ex in enumerate(examples_val):
        by_battle[ex["battle_id"]].append(i)

    if battle_id is None:
        battle_id = rng.choice(list(by_battle.keys()))

    idxs = sorted(
        by_battle.get(battle_id, []),
        key=lambda i: (examples_val[i].get("turn_number", 10**9), i),
    )
    if not idxs:
        print(f"No examples found for battle_id={battle_id}")
        return

    # ----- choose start so we have runway -----
    L = len(idxs)
    if L <= max_turns:
        start_pos = 0
    else:
        # pick start_pos in [0, L - max_turns] inclusive
        start_pos = int(rng.integers(0, (L - max_turns) + 1))
    # -----------------------------------------

    start_i = idxs[start_pos]
    start_player = examples_val[start_i].get("player", "p1")

    other_player = "p2" if start_player == "p1" else "p1"
    expected = start_player

    # Build an alternating window scanning forward
    window = []
    for j in range(start_pos, L):
        i = idxs[j]
        pl = examples_val[i].get("player", None)
        if pl == expected:
            window.append(i)
            expected = other_player if expected == start_player else start_player
            if len(window) >= max_turns:
                break

    # If alternation prevented us from filling max_turns (rare but possible),
    # fall back to just taking the next max_turns examples in order.
    if len(window) < max_turns:
        window = idxs[start_pos : min(L, start_pos + max_turns)]

    logits_batch = model.predict(X_val[window], verbose=0)

    print("\n========================================")
    print(f"Battle: {battle_id}")
    print(f"Replay: https://replay.pokemonshowdown.com/{battle_id}")
    print(f"Start:  idx_in_battle={start_pos}/{L-1}  (player={start_player})")
    print(f"Mode:   alternating players (fallback-to-sequential if needed)")
    print(f"Showing {len(window)} steps (max_turns={max_turns})")
    print("========================================")

    for row_i, local_i in enumerate(window):
        ex = examples_val[local_i]
        logits = logits_batch[row_i].astype(np.float32)

        turn_number = ex.get("turn_number", "?")
        perspective = ex.get("player", "?")
        state = ex.get("state", {}) or {}

        p1_uid, p1_species, p1_obs = _get_active_info(state, "p1")
        p2_uid, p2_species, p2_obs = _get_active_info(state, "p2")

        print("\n------------------------------")
        print(f"Turn:   {turn_number} (perspective={perspective})")
        print(f"Facing: {p1_species or '?'} ({p1_uid or '?'})  vs  {p2_species or '?'} ({p2_uid or '?'})")
        if show_observed_moves:
            if p1_uid:
                print(f"p1 active observed_moves: {p1_obs}")
            if p2_uid:
                print(f"p2 active observed_moves: {p2_obs}")

        topk_idx = np.argsort(logits)[-k:][::-1]

        print("\nPredictions (logits):")
        for rank, ci in enumerate(topk_idx, start=1):
            mv = idx_to_move.get(int(ci), "<UNK>")
            print(f"  {rank}. {mv:20s} logit={float(logits[ci]):.3f}")

        if show_softmax:
            probs = _softmax(logits)
            print("\nPredictions (softmax over all moves):")
            for rank, ci in enumerate(topk_idx, start=1):
                mv = idx_to_move.get(int(ci), "<UNK>")
                print(f"  {rank}. {mv:20s} p={float(probs[ci]):.3f}")

        if y_val is not None:
            true_idx = int(y_val[local_i])
            true_move = idx_to_move.get(true_idx, "<UNK>")
            true_rank = int(np.sum(logits > logits[true_idx]) + 1)
            print(f"\nTrue move: {true_move} (rank {true_rank}, logit={float(logits[true_idx]):.3f})")

        if "action" in ex:
            print(f"Logged action: {ex['action']}")

show_battle_window_alternating(
    model,
    X_val,
    y_val,
    examples_val,
    max_turns=20,
    k=5,
    seed=100,
)


In [ ]:
import numpy as np

idx_to_move = {i: m for m, i in move_vocab.items()}

def softmax_rows(logits_2d):
    x = logits_2d - np.max(logits_2d, axis=1, keepdims=True)
    e = np.exp(x)
    return e / np.sum(e, axis=1, keepdims=True)

def show_predictions_probs(model, X, y=None, n=5, k=3, seed=0):
    rng = np.random.default_rng(seed)
    n = min(n, len(X))
    sample_idxs = rng.choice(len(X), size=n, replace=False)

    logits = model.predict(X[sample_idxs], verbose=0)
    probs = softmax_rows(logits)

    for row_i, ex_i in enumerate(sample_idxs):
        p = probs[row_i]
        topk_idx = np.argsort(p)[-k:][::-1]
        topk = [(idx_to_move.get(int(ci), "<UNK>"), float(p[ci])) for ci in topk_idx]

        print(f"\nExample idx={ex_i}")
        print(f"Top-1: {topk[0][0]}  (prob={topk[0][1]:.3f})")
        print(f"Top-{k}:")
        for rank, (mv, pr) in enumerate(topk, start=1):
            print(f"  {rank}. {mv:20s} prob={pr:.3f}")

        if y is not None:
            true_idx = int(y[ex_i])
            true_move = idx_to_move.get(true_idx, "<UNK>")
            true_rank = int(np.sum(p > p[true_idx]) + 1)
            print(f"True:  {true_move}  (rank {true_rank}, prob={float(p[true_idx]):.3f})")

# call it:
show_predictions_probs(model, X_val, y=y_val, n=5, k=3, seed=42)
# Persist trained artifacts for API use
model.save("model.keras")

import json
with open("move_vocab.json", "w") as f:
    json.dump(move_vocab, f)

print("Saved model.h5 and move_vocab.json")

In [ ]:
logits = model.predict(X_val, verbose=0)
top1 = np.argmax(logits, axis=1)
from collections import Counter
c = Counter(top1)
for cls, cnt in c.most_common(15):
    print(idx_to_move.get(cls, "<UNK>"), cnt)


In [ ]:
# Encoding static pokemon data

In [ ]:
from StateVectorization import vectorize_dataset_static
dex = StaticDex.from_source(local_path="./pokedex.json")  # or local path to pokedex.json

X_train_dict, y_train = vectorize_dataset_static(X_train, move_vocab, dex)
X_val_dict, y_val = vectorize_dataset_static(X_val, move_vocab, dex)

# history = model.fit(
#     X_train_dict, y_train,
#     validation_data=(X_val_dict, y_val),
#     epochs=20,
#     batch_size=256,
#     verbose=1,
# )


In [ ]:
# New model considering the added input: from tensorflow import keras
from tensorflow.keras import layers

num_input = layers.Input(shape=(input_dim_extended,), name="num")

my_species = layers.Input(shape=(), dtype="int32", name="my_species")
op_species = layers.Input(shape=(), dtype="int32", name="op_species")
my_t1 = layers.Input(shape=(), dtype="int32", name="my_t1")
my_t2 = layers.Input(shape=(), dtype="int32", name="my_t2")
op_t1 = layers.Input(shape=(), dtype="int32", name="op_t1")
op_t2 = layers.Input(shape=(), dtype="int32", name="op_t2")

# embeddings (sizes are small on purpose)
sp_emb = layers.Embedding(input_dim=len(species_to_idx), output_dim=24, name="sp_emb")
ty_emb = layers.Embedding(input_dim=len(type_to_idx), output_dim=6, name="ty_emb")

embs = [
    sp_emb(my_species), sp_emb(op_species),
    ty_emb(my_t1), ty_emb(my_t2),
    ty_emb(op_t1), ty_emb(op_t2),
]
emb_cat = layers.Concatenate()(embs)
emb_cat = layers.Flatten()(emb_cat)

x = layers.Concatenate()([num_input, emb_cat])
x = layers.Dense(256, activation="relu")(x)
x = layers.Dropout(0.1)(x)
x = layers.Dense(256, activation="relu")(x)
x = layers.Dropout(0.1)(x)
logits = layers.Dense(num_classes)(x)

model = keras.Model(
    inputs=[num_input, my_species, op_species, my_t1, my_t2, op_t1, op_t2],
    outputs=logits,
)

model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=[
        keras.metrics.SparseCategoricalAccuracy(name="top1"),
        keras.metrics.SparseTopKCategoricalAccuracy(k=3, name="top3"),
    ],
)


In [ ]:
# Setup RNN to embed state each turn as input
import numpy as np
from collections import defaultdict

def build_player_sequences_from_indices(
    examples,
    X,
    y,
    indices,
    pad_value=0.0,
):
    """
    Build sequences grouped by (battle_id, player). Each sequence is time-ordered by turn_number.

    examples: list[dict] aligned with X and y
    X: np.ndarray shape (N, D)
    y: np.ndarray shape (N,)
    indices: iterable of ints (e.g., train_idx or val_idx)
    pad_value: value used to pad X

    Returns:
      X_seq: (S, T_max, D) float32
      y_seq: (S, T_max) int64  (padded positions filled with 0, but masked by sw)
      sw_seq:(S, T_max) float32 (1 for real timestep, 0 for padding)
      meta:  list of dict metadata per sequence
    """
    # 1) group indices by (battle_id, player)
    groups = defaultdict(list)
    for i in indices:
        ex = examples[i]
        bid = ex["battle_id"]
        pl = ex["player"]
        tn = ex.get("turn_number", ex.get("turn_index", 0))
        groups[(bid, pl)].append((tn, i))

    # 2) sort each group by turn_number
    seqs = []
    meta = []
    for (bid, pl), items in groups.items():
        items.sort(key=lambda t: (t[0], t[1]))  # (turn_number, stable index)
        seq_idxs = [i for _, i in items]
        seqs.append(seq_idxs)
        meta.append({"battle_id": bid, "player": pl, "length": len(seq_idxs)})

    if not seqs:
        raise ValueError("No sequences produced. Check indices/examples alignment.")

    # 3) pad to max length
    D = X.shape[1]
    T_max = max(len(s) for s in seqs)
    S = len(seqs)

    X_seq = np.full((S, T_max, D), pad_value, dtype=np.float32)
    y_seq = np.zeros((S, T_max), dtype=np.int64)
    sw_seq = np.zeros((S, T_max), dtype=np.float32)

    for s_i, seq_idxs in enumerate(seqs):
        t_len = len(seq_idxs)
        X_seq[s_i, :t_len, :] = X[seq_idxs]
        y_seq[s_i, :t_len] = y[seq_idxs]
        sw_seq[s_i, :t_len] = 1.0

    return X_seq, y_seq, sw_seq, meta


In [ ]:
# X and y already aligned with examples_both
# train_idx, val_idx from group_split_by_battle_id(examples_both, ...)

X_train_seq, y_train_seq, sw_train_seq, meta_train = build_player_sequences_from_indices(
    examples_both, X, y, train_idx
)
X_val_seq, y_val_seq, sw_val_seq, meta_val = build_player_sequences_from_indices(
    examples_both, X, y, val_idx
)

print("Train seq:", X_train_seq.shape, y_train_seq.shape, sw_train_seq.shape)
print("Val seq:  ", X_val_seq.shape, y_val_seq.shape, sw_val_seq.shape)
print("Example meta:", meta_train[0])


In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

def build_gru_policy(input_dim, num_classes, rnn_units=128, dropout=0.1):
    inp = layers.Input(shape=(None, input_dim), name="state_seq")  # variable T
    x = layers.Masking(mask_value=0.0)(inp)  # matches pad_value above
    x = layers.GRU(rnn_units, return_sequences=True)(x)
    x = layers.Dropout(dropout)(x)
    x = layers.GRU(rnn_units, return_sequences=True)(x)
    x = layers.Dropout(dropout)(x)
    logits = layers.TimeDistributed(layers.Dense(num_classes), name="logits")(x)
    return keras.Model(inp, logits)

model_rnn = build_gru_policy(
    input_dim=X.shape[1],
    num_classes=len(move_vocab),
    rnn_units=128,
    dropout=0.1,
)

model_rnn.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=[keras.metrics.SparseCategoricalAccuracy(name="top1")],
)

model_rnn.summary()


In [ ]:
# Persist trained artifacts for API use
model.save("model.h5")

import json
with open("move_vocab.json", "w") as f:
    json.dump(move_vocab, f)

print("Saved model.h5 and move_vocab.json")